# Filter biller.json by Billers

This notebook reads `src/biller.json`, groups entries by the `Billers` column, and writes one JSON file per category into `src/biller_categories/`.

In [2]:
import json
import re
from pathlib import Path

BASE_DIR = Path.cwd()
INPUT_FILE = BASE_DIR / "src" / "biller.json"
OUTPUT_DIR = BASE_DIR / "src" / "biller_categories"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with INPUT_FILE.open("r", encoding="utf-8") as f:
    billers = json.load(f)


def slugify(value: str) -> str:
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_") or "uncategorized"


def normalize_category(value: str) -> str:
    return re.sub(r"\s+", " ", value).strip()


# Remove stale outputs from previous runs.
for existing_file in OUTPUT_DIR.glob("*.json"):
    existing_file.unlink()

by_slug: dict[str, dict] = {}
for item in billers:
    raw_category = item.get("Billers") or "Uncategorized"
    category = normalize_category(raw_category) or "Uncategorized"
    slug = slugify(category)

    if slug not in by_slug:
        by_slug[slug] = {"category": category, "items": []}
    by_slug[slug]["items"].append(item)

summary = []
for slug, payload in sorted(by_slug.items(), key=lambda x: x[1]["category"].lower()):
    category = payload["category"]
    items = payload["items"]
    file_name = f"{slug}.json"
    file_path = OUTPUT_DIR / file_name

    with file_path.open("w", encoding="utf-8") as f:
        json.dump(items, f, indent=2, ensure_ascii=False)

    summary.append(
        {
            "category": category,
            "count": len(items),
            "file": str(file_path.relative_to(BASE_DIR)),
        }
    )

summary_path = OUTPUT_DIR / "_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Created {len(summary)} category files in {OUTPUT_DIR}")
for row in summary:
    print(f"- {row['category']}: {row['count']} -> {row['file']}")

Created 29 category files in /Users/shubhamgoswami/.gemini/antigravity/scratch/electricity-sms-generator/src/biller_categories
- Agent Collection: 4 -> src/biller_categories/agent_collection.json
- Broadband Postpaid: 394 -> src/biller_categories/broadband_postpaid.json
- Cable TV: 69 -> src/biller_categories/cable_tv.json
- Clubs and Associations: 34 -> src/biller_categories/clubs_and_associations.json
- Credit Card: 35 -> src/biller_categories/credit_card.json
- Donation: 83 -> src/biller_categories/donation.json
- DTH: 6 -> src/biller_categories/dth.json
- eChallan: 4 -> src/biller_categories/echallan.json
- Education Fees: 20590 -> src/biller_categories/education_fees.json
- Electricity: 107 -> src/biller_categories/electricity.json
- EV Recharge: 3 -> src/biller_categories/ev_recharge.json
- Fastag: 30 -> src/biller_categories/fastag.json
- Fleet Card Recharge: 2 -> src/biller_categories/fleet_card_recharge.json
- Gas: 39 -> src/biller_categories/gas.json
- Housing Society: 365 ->